---   
 <img align="left" width="75" height="75"  src="https://upload.wikimedia.org/wikipedia/en/c/c8/University_of_the_Punjab_logo.png"> 

<h1 align="center">Department of Data Science</h1>

---
<h3><div align="right">Instructor: Muhammad Arif Butt, Ph.D.</div></h3>    

<br><br>
<h1 align="center">Lec-14: Multi-Modality of AI Models (Part-II)</h1>

# Learning agenda of this notebook
1. Speech Recognition: Audio → Text
    - OpenAI's Whisper
    - Using OpenAI's whisper-base with Transformers Pipeline
2. Speech Generation: Text → Speech
    - Using OpenAI's tts-1
    - Using Google Text-to-Speech (gTTS)
    - Using Open-Source Models with Transformers Pipeline
3. Project (Cross-Modal Chaining)
    - Speak: “Describe this picture of a cat.”
    - Model transcribes voice (Whisper).
    - Model analyzes the image (GPT-4o).
    - Model generates a narration in text.
    - Model speaks it back (TTS).
4. Text → Video Generation
5. Video → Text (Video Understanding)

In [1]:
#  Load the API  API Keys
import os
from dotenv import load_dotenv
load_dotenv('../keys/.env', override=True) 

openai_api_key = os.getenv('OPENAI_API_KEY')
anthropic_api_key = os.getenv('ANTHROPIC_API_KEY')
google_api_key = os.getenv('GOOGLE_API_KEY')
groq_api_key = os.getenv("GROQ_API_KEY")
hf_token = os.getenv('HF_TOKEN')

if openai_api_key:
    print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
else:
    print("OpenAI API Key not set")
    
if anthropic_api_key:
    print(f"Anthropic API Key exists and begins {anthropic_api_key[:8]}")
else:
    print("Anthropic API Key not set")

if google_api_key:
    print(f"Google API Key exists and begins {google_api_key[:8]}")
else:
    print("Google API Key not set (and this is optional)")

if groq_api_key:
    print(f"Groq API Key exists and begins {groq_api_key[:8]}")
else:
    print("Groq API Key not set")

if hf_token:
    print(f"Hugging Face Tokens exists and begins {hf_token[:8]}")
else:
    print("Hugging Face tokens not set")


OpenAI API Key exists and begins sk-proj-
Anthropic API Key exists and begins sk-ant-a
Google API Key exists and begins AIzaSyDA
Groq API Key exists and begins gsk_LyFp
Hugging Face Tokens exists and begins hf_oEyHP


# 1. <span style='background :lightgreen' >Speech Recognition: Audio → Text</span>

<h2 align="center"><div class="alert alert-success" color=magenta style="margin: 20px">Audio-to-text models use automatic speech recognition (ASR) to transcribe spoken language into written text, supporting real-time transcription, multilingual speech, and speaker-specific insights for accessibility, productivity, and audio analysis.</div></h2>

### Closed Source:
- **OpenAI Whisper API:** Hosted version of Whisper offering accurate multilingual transcription and translation through OpenAI’s API. (https://openai.com/research/whisper)
- **Google Speech-to-Text:** Cloud service offering real-time transcription, speaker diarization, and support for over 125 languages. (https://cloud.google.com/speech-to-text)
- **Azure Speech Services:** Microsoft’s enterprise-ready speech recognition with customizable models and integration into Azure AI. (https://azure.microsoft.com/en-us/products/ai-services/speech-to-text)
### Open Source:
- **Whisper (local):** Open-source ASR (Automatic Speech Recognition) model by OpenAI supporting multilingual transcription and translation. (https://github.com/openai/whisper)
- **Wav2Vec2:** Facebook AIs self-supervised speech representation model, fine-tuned for high-quality ASR (Automatic Speech Recognition). (https://huggingface.co/facebook/wav2vec2-base-960h)
- **SpeechT5:** Microsoft’s transformer-based model handling ASR (Automatic Speech Recognition), TTS (Text-to-Speech), and speech translation. (https://huggingface.co/microsoft/speecht5_asr)

## a.  OpenAI's Whisper:
- The **OpenAI's Whisper** model is most commonly used for transcribing audio recordings, generating subtitles for videos, converting spoken content into written documents.
- This code uses OpenAI’s hosted whisper-1 model (a closed-source, cloud-run version of Whisper) to transcribe a local audio file by sending it to OpenAI’s API.
- OpenAI client provides two endpoints for the Whisper model:
    - **`audio.transcriptions.create()`:** Converts speech → text in the same language. Performs automatic speech recognition (ASR). Example: English audio → English text, Urdu audio → Urdu text and so on
    - **`audio.translations.create()`:** Converts speech (in any language) → English text. Example: Urdu audio → English text (Cannot output in language other than English as of today)
- It detects the language automatically and produces text in the language spoken in the audio. Has support of 90+ languages like English (en), Urdu (ur), Bengali (bn), Arabic (ar), French (fr), German (de), Japanese (ja), Hindi (hi), Hebrew (he) and so on
- Practically you can use this model to transcribe audio into whatever language the audio is in or to translate and transcribe the audio into English.

### Example 1 (Generating Transcriptions with `whisper-1`)

In [2]:
# Play an English audio file, whose transcription we want to create
from IPython.display import Audio
Audio('../data/audios/arif-english-audio.mp3', autoplay=False)

In [3]:
# Converts English speech to English Text
from openai import OpenAI

client = OpenAI(api_key=openai_api_key) # Although open source the 'whisper-1' model do require API key because it calls OpenAI’s hosted service.

audio_file = open('../data/audios/arif-english-audio.mp3', 'rb')
transcript = client.audio.transcriptions.create(
    model="whisper-1",           # REQUIRED: Only "whisper-1" is supported by the OpenAI transcription endpoint.Only "whisper-1" is supported by the OpenAI transcription endpoint.
    file=audio_file,             # REQUIRED: The audio file you want to transcribe (mp3, mp4, m4a, wav). Must be opened in binary mode
    response_format="text",      # Default is json, other values can be srt (returns SubRip Text format used by video players), vtt (returns Web Video Text Tracks format (used widely on websites)
    temperature=0,               # Controls randomness in Whisper’s decoding. Default: 0. Usually left at 0 for best accuracy.
    prompt = None,               # A text hint to guide the transcription, e.g., "This audio contains medical terminology"
    language=None                # Whisper normally auto-detects the audio language, but you can manually specify the language of the audio (en,
)
audio_file.close()
transcript

'Assalam-o-Alaikum Students, I welcome you all to Learning AI with Arif Butt.\n'

In [1]:
# Play an Urdu audio file, whose transcription we want to create
from IPython.display import Audio
Audio('../data/audios/arif-urdu-audio.mp3', autoplay=False)

In [4]:
# Converts Urdu speech to Urdu Text
from openai import OpenAI

client = OpenAI(api_key=openai_api_key) # Although open source the 'whisper-1' model do require API key because it calls OpenAI’s hosted service.


audio_file = open('../data/audios/arif-urdu-audio.mp3', 'rb')
transcript = client.audio.transcriptions.create(
    model="whisper-1",           # REQUIRED: Only "whisper-1" is supported by the OpenAI transcription endpoint.Only "whisper-1" is supported by the OpenAI transcription endpoint.
    file=audio_file,             # REQUIRED: The audio file you want to transcribe (mp3, mp4, m4a, wav). Must be opened in binary mode
    response_format="text",      # Default is json, other values can be srt (returns SubRip Text format used by video players), vtt (returns Web Video Text Tracks format (used widely on websites)
    temperature=0,               # Controls randomness in Whisper’s decoding. Default: 0. Usually left at 0 for best accuracy.
    prompt = None,               # A text hint to guide the transcription, e.g., "This audio contains medical terminology"
    language=None                # Whisper normally auto-detects the audio language, but you can manually specify the language of the audio (en,
)
audio_file.close()
transcript

'عزیز طلبہ السلام علیکم میں آپ سب کو جنریٹیو آرٹیفیشل انٹیلیجنز کے کورس میں خوشحامدید کہتا ہوں\n'

In [5]:
# Converts Urdu speech to Hindi Text (same audio file as above, shown with an explicit base_url)
from openai import OpenAI

client = OpenAI(base_url="https://api.openai.com/v1", api_key=openai_api_key) # Although open source the 'whisper-1' model do require API key because it calls OpenAI’s hosted service.

audio_file = open('../data/audios/arif-urdu-audio.mp3', 'rb')
transcript = client.audio.transcriptions.create(
    model="whisper-1",           # REQUIRED: Only "whisper-1" is supported by the OpenAI transcription endpoint.Only "whisper-1" is supported by the OpenAI transcription endpoint.
    file=audio_file,             # REQUIRED: The audio file you want to transcribe (mp3, mp4, m4a, wav). Must be opened in binary mode
    response_format="text",      # Default is json, other values can be srt (returns SubRip Text format used by video players), vtt (returns Web Video Text Tracks format (used widely on websites)
    temperature=0,               # Controls randomness in Whisper’s decoding. Default: 0. Usually left at 0 for best accuracy.
    prompt = None,               # A text hint to guide the transcription, e.g., "This audio contains medical terminology"
    language="hi"                # Whisper normally auto-detects the audio language, but you can manually specify the language of the audio (en,
)
audio_file.close()
transcript

'अजीज तलबा अस्सलाम ओनिकम, मैं आप सबको जेनेरेटिव आर्टिफिशल इंटेलिजन्स के कोर्स में खुशामदीत कहता हूं।\n'

### Example 2 (Doing Translations with `whisper-1`)
- OpenAI’s Whisper can take audio in many languages and **translate** it into English text.
- It detects the spoken language automatically and produces an English transcription of the audio.

In [6]:
# Play an Urdu audio file, which we want to translate/transcribe in English
from IPython.display import Audio
Audio('../data/audios/arif-urdu-audio.mp3', autoplay=False)

In [7]:
# Translates Urdu speech to English Text
from openai import OpenAI

client = OpenAI(api_key=openai_api_key)

audio_file = open('../data/audios/arif-urdu-audio.mp3', 'rb')
result = client.audio.translations.create(
        model="whisper-1",
        file=audio_file
    )
audio_file.close()
print(result.text) 

Azeez Talba, Assalam-o-Alaikum. I welcome you all to the course on Generative Artificial Intelligence.


In [8]:
# Play a German language audio file, whose English transcription we want to create
from IPython.display import Audio
Audio('../data/audios/german-audio.mp3', autoplay=False)

In [9]:
# Translates German speech to English Text
from openai import OpenAI
import os
from dotenv import load_dotenv

load_dotenv('../keys/.env', override=True) 
openai_api_key = os.getenv('OPENAI_API_KEY')

client = OpenAI(api_key=openai_api_key)

audio_file = open('../data/audios/german-audio.mp3', 'rb')
result = client.audio.translations.create(
        model="whisper-1",
        file=audio_file
    )
audio_file.close()
print(result.text) 

Hello? Great, dad! It's really great here with the lions. Only this lion here, your BQ, is not here. He's probably sleeping inside, isn't he? Not today. Because today the lion is not inside, but... ...not here at all. But? At the doctor. Once a year he is examined, vaccinated, something like that. I understand. But now tell me, how was it at school? Good. The patch on your forehead, where did it come from? At the school yard. Doesn't matter. Really, dad, doesn't matter. No, it doesn't matter. They said things about you.


## b.  OpenAI's `whisper-base` (Open Source):
- The **`openai/whisper-base`** is an open source multilingual speech recognition model that can be run locally without requiring API calls. It's commonly used for transcribing audio recordings, generating subtitles for videos, and converting spoken content into written documents.
- The model comes in different sizes: tiny, base, small, medium, and large, with varying accuracy and computational requirements. The whisper-base model offers a good balance between speed and accuracy for most use cases.
- The following code when run for the first time will download `openai/whisper-base` model from Hugging Face. The model files are downloaded once and cached for future use.
- The Transformers pipeline provides flexible configuration through the generate_kwargs parameter:
    - task="transcribe": Converts speech → text in the same language. Performs automatic speech recognition (ASR). Example: English audio → English text, Urdu audio → Urdu text and so on
    - task="translate": Converts speech (in any language) → English text only. Example: Urdu audio → English text (Cannot output in language other than English)
- It can automatically detect the language or you can explicitly specify it using language parameter. Supports 90+ languages including English (en), Urdu (ur), Bengali (bn), Arabic (ar), French (fr), German (de), Japanese (ja), Hindi (hi), Hebrew (he) and so on
- HuggingFace: https://huggingface.co/openai/whisper-base
- GitHub Repo: https://github.com/openai/whisper

### Example 1 (Generating Transcriptions with `whisper-base`)

In [10]:
# Play an English audio file, whose transcription we want to create
from IPython.display import Audio
Audio('../data/audios/arif-english-audio.mp3', autoplay=True)

In [11]:
# Converts English speech to English Text (You may have to install ffmpeg binary on your OS for this code to work as ffmpeg is required to load audio files from filename
from transformers import pipeline

# Load ASR (Automatic Speech Recognition) pipeline
pipe = pipeline(task="automatic-speech-recognition", model="openai/whisper-base", device=-1)

# Transcribe the speech to text. The pipeline can accept either a file path or audio data directly
transcript = pipe('../data/audios/arif-english-audio.mp3',
                    generate_kwargs={
                                    "language": "english",  # Specify language of the input audio (can use "urdu", "hindi", etc.)
                                    "task": "transcribe",  # Options: "transcribe" or "translate"
                                    "temperature": 0.0,  # Lower = more deterministic output
                                    }
                    )
# Display the results
print(f"Transcribed Text: {transcript['text']}")

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensLogitsProcessor'> to see related `.generate()` flags.
[transformers] A custom logits processor of type <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> has been passed to `.generate()`, but it was also created in `.generate()`, given its parameterization. The custom <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> will take precedence. Please check the docstring of <class 'transformers.generation.logits_process.SuppressTokensAtBeginLogitsProcessor'> to see related `.generate()` flags.
[

Transcribed Text:  Assalamu alaikum students, I welcome you all to learning AI with RFBUT.


In [12]:
# Play an Urdu audio file, whose transcription we want to create
from IPython.display import Audio
Audio('../data/audios/arif-urdu-audio.mp3', autoplay=True)

In [13]:
# Converts Urdu speech to Urdu Text
from transformers import pipeline

# Load ASR (Automatic Speech Recognition) pipeline
pipe = pipeline("automatic-speech-recognition", "openai/whisper-base", device=-1)

# Transcribe the speech to text. The pipeline can accept either a file path or audio data directly
transcript = pipe('../data/audios/arif-urdu-audio.mp3',
    generate_kwargs={
        "language": "urdu",  # Specify language of the input audio (can use "urdu", "hindi", etc.)
        "task": "transcribe",  # Options: "transcribe" or "translate"
        "temperature": 0.0,  # Lower = more deterministic output
    }
)

# Display the results
print(f"Transcribed Text: {transcript['text']}")

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Transcribed Text:  ازیز طلبہ اصلام علیکم میں آپ سک کو جنریٹیف ارٹیفشل انٹیلیجنس کے کورس میں خوشام دیت کہتا ہوں


### Example 2 (Doing Translations with `whisper-base`)

In [14]:
# Play an Urdu audio file, whose English translation we want to generate
from IPython.display import Audio
Audio('../data/audios/arif-urdu-audio.mp3', autoplay=False)

In [16]:
# Translates Urdu speech to English Text
from transformers import pipeline

# Load ASR (Automatic Speech Recognition) pipeline
pipe = pipeline("automatic-speech-recognition", "openai/whisper-base", device=-1)

# Translate the speech to English text
result = pipe('../data/audios/arif-urdu-audio.mp3',
    generate_kwargs={
        "language": "urdu",      # Specify source language as Urdu
        "task": "translate",     # Change to "translate" for translation to English
        "temperature": 0.2,      # Lower = more deterministic output
    }
)

# Display the results
print(f"Translated Text: {result['text']}")

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

Translated Text:  Assalam-o-Alaikum. I am your representative artificial intelligence course.


<img align=right src="../images/pl6.png" width="1000">

# 2. <span style='background :lightgreen' >Speech Generation: Text → Speech</span>

Text-to-Speech generation enables AI systems to convert written input into natural-sounding speech or even music. These models can produce lifelike voices, clone speaker styles, or generate musical compositions, opening up applications in accessibility, entertainment, and personalized content creation.
- **Closed Source:**
    - **OpenAI TTS:** OpenAI’s high-quality text-to-speech model producing natural, expressive voices in real time. (https://openai.com/index/tts/)
    - **ElevenLabs:** Industry-leading AI voice platform offering lifelike speech synthesis, voice cloning, and multilingual TTS. (https://elevenlabs.io/)
    - **Microsoft Azure Speech:** Cloud service for TTS with customizable voices, neural synthesis, and enterprise integration. (https://azure.microsoft.com/en-us/products/ai-services/text-to-speech)
- **Open Source:**
    - **Bark:** Suno’s transformer-based model for realistic TTS with non-verbal sounds, music, and multilingual capabilities. (https://github.com/suno-ai/bark)
    - **Tortoise TTS:** Open-source model focusing on ultra-realistic speech generation and voice cloning, though slower at inference. (https://github.com/neonbjb/tortoise-tts)
    - **XTTS-v2:** Coqui’s cross-lingual TTS model supporting multiple languages and speaker adaptation with open weights. (https://huggingface.co/coqui/XTTS-v2)

### Using OpenAI's `tts-1`
- **TTS-1** is OpenAI’s hosted text-to-speech model that converts written text into natural-sounding speech. It is not open-source and is available only through the OpenAI API.
- It can generate speech in multiple languages, including English, French, German, Spanish, Chinese, Japanese, Korean, Russian, and many more.
- The model supports expressive, human-like voices, including different speaking styles and tones depending on the selected voice preset.
- TTS-1 can be used for high-quality voiceovers, narration, interactive assistants, and real-time audio output (possible through streaming).
- The API is useful for:
    - Creating audiobooks, e-learning content, and multilingual educational material
    - Building voice interfaces for chatbots and applications
    - Generating narration for videos, podcasts, tutorials, and digital media content
    - Developing accessibility tools for individuals with low vision or reading difficulties
    - Enhancing games and immersive experiences (e.g., VR narration, character voices)
    - Helps with language learning, offering pronunciation guides or listening practice in a natural-sounding voice.
- OpenAI client provides two main endpoints for the TTS-1 model:
    - **`audio.speech.create()`:** Converts text → speech in the same language as the input text. Performs high-quality text-to-audio synthesis using lifelike voices. Example: English text → English speech, French text → French speech, Chinese text → Chinese speech. Note: Output language always matches the input text language (no automatic translation).
    - **`audio.speech.with_streaming_response.create()`:** Provides real-time streaming text → speech, allowing applications to start playing generated audio before the full synthesis is complete. Ideal for interactive assistants, live narration, or quick audio previews. Same behavior as above regarding language: the model reads the text as-is and does not translate.

In [17]:
# Using audio.speech.create()
from IPython.display import Audio 
from openai import OpenAI

client = OpenAI(api_key=openai_api_key)   

# Generte audio of above text in English language using `client.audio.speech.create()` method which returns raw audio bytes
response = client.audio.speech.create(
                             model="tts-1",                                                               # Required: can call tts-1-hd for better quality
                             input="Hello students, welcome to learning Generative AI with Arif Butt.",   # Required: The text you want converted into speech. Can be string or array of text segments.    
                             voice="alloy"                                                                # Specifies which voice to use. Available voices: alloy, verse, coral, onxy, nova, shimmer, etc.
                            )

# Save the audio in a file named tts.mp3
with open('../data/audios/tts-generated-audio1.mp3', 'wb') as f:
    f.write(response.content)

# Play the audio in the notebook
Audio(response.content)

In [18]:
# Using audio.speech.create()
from IPython.display import Audio 
from openai import OpenAI

client = OpenAI(api_key=openai_api_key)   

# Generte audio of above text in English language using `client.audio.speech.create()` method which returns raw audio bytes
response = client.audio.speech.create(
                             model="tts-1",                                                              
                             input="پیارے طلباء، میں آپ سب کو جنریٹو آرٹیفیشل انٹیلی جنس کے کورس میں خوش آمدید کہتا ہوں۔ میں آپ کا انسٹرکٹر عارف بٹ ہوں۔",   
                             voice="alloy"                                                             
                            )

# Save the audio in a file named tts.mp3
with open('../data/audios/tts-generated-audio2.mp3', 'wb') as f:
    f.write(response.content)

# Play the audio in the notebook
Audio('../data/audios/tts-generated-audio2.mp3') # loads and plays audio from a saved file on disk

### Using `audio.speech.with_streaming_response.create()`
- The TTS model starts generating audio immediately
- Audio is sent back in small byte chunks
- Your code receives each chunk as soon as it’s ready
- You don’t wait for the full audio before receiving data
- This is ideal for:
    - Real-time assistants
    - Low-latency voice apps
    - Live narration

In [2]:
# Using audio.speech.with_streaming_response.create()
from IPython.display import Audio
from openai import OpenAI

client = OpenAI(base_url="https://api.openai.com/v1", api_key=openai_api_key)

# Stream the audio from tts-1
with client.audio.speech.with_streaming_response.create(
                                                    model="tts-1",
                                                    input="Dr. Muhammad Arif Butt is an accomplished Assistant Professor at the Department of Data Science, University of the Punjab (PU), Lahore, Pakistan. He holds an MSc and MPhil (both with Gold Medals) and a Ph.D. in Computer Science from PUCIT, University of the Punjab. His research focuses on fuzzy inference models applied to operating systems, embedded systems, and cloud-based services, particularly in decision-making under uncertain and imprecise conditions.With over 33 years of experience in teaching and management, Dr. Butt has served in both the Pakistan Army and University of the Punjab, bringing a wealth of interdisciplinary expertise. His teaching specializations include embedded and real-time operating systems, system programming, cybersecurity, and artificial intelligence.Beyond academia, he is a technology entrepreneur, serving as the Founder of Excaliat and Falcon-Hunt and Co-Founder of Tbox Solutionz. In recent years, he has gained significant expertise in vulnerability research, binary exploitation, and exploit development, excelling in identifying and mitigating critical security risks across diverse platforms. His deep understanding of software architectures, memory corruption techniques, and attack vectors strengthens his ability to proactively enhance cybersecurity defences.A dedicated and results-driven professional, Dr. Butt is recognized for his strong organizational skills, strategic thinking, and ability to thrive in collaborative environments. His expertise in cybersecurity and emerging technologies enables him to contribute effectively to both academic research and industry innovation, reinforcing defences against evolving cyber threats.",
                                                    voice="alloy",
    
                                            ) as resp:
                                                    audio_bytes = b""               # collect bytes as they arrive
                                                    for chunk in resp.iter_bytes(): # iter_bytes() yields small audio chunks as soon as they are generated by the model (real-time streaming)
                                                        audio_bytes += chunk

# Play the streamed audio immediately (no saving)
Audio(audio_bytes, autoplay=True)     # plays in-memory audio bytes (ideal for streaming)

### Using Google Text-to-Speech (gTTS)
- **gTTS (Google Text-to-Speech)** is an open-source Python library that interfaces with Google Translate’s TTS API to convert written text into natural-sounding speech.
- It is open-source and free to use, but relies on Google’s hosted TTS service. It sends text to Google Translate’s TTS endpoint and returns audio
- It supports multiple languages, including English, French, German, Spanish, Chinese, Japanese, Korean, Russian, Hindi, Urdu, Arabic, and many more.
- gTTS can produce different accents and regional pronunciations by adjusting the tld (top-level domain) parameter, e.g., 'com' → American, 'co.uk' → British, 'co.in' → Indian.
- The library allows controlling speaking speed via the slow parameter (True for slow speech, False for normal).
- gTTS is suitable for:
    - Generating voiceovers for videos, tutorials, and presentations
    - Creating educational content and language learning material
    - Developing accessibility tools for people with low vision or reading difficulties
    - Narrating articles, e-books, or interactive applications
    - Producing multilingual audio content quickly and efficiently
- gTTS usage workflow:
    - gTTS() object creation: Prepares the text, selects language, accent, and speech rate. Does not generate audio immediately.
    - save(filename) method: Sends the text to Google TTS API and saves the resulting audio as an MP3 file.
    - Audio playback: Can be played in Python using libraries like IPython.display.Audio() or any MP3 player.
- **Notes:**
    - Unlike OpenAI’s tts-1, gTTS does not provide streaming playback; the full audio must be generated before it can be played.
    - gTTS does not offer multiple expressive voices or styles; the voice is determined by Google’s TTS engine and tld.

In [3]:
from gtts import gTTS              # gTTS: Library that uses Google Translate's TTS API Hosted at 'https://translate.google.co.in/' 
from IPython.display import Audio  # Audio: To play audio inside Jupyter Notebook

#  Input text you want to convert to speech
text = "Hello students, welcome to learning Generative AI with Arif Butt."

# The gTTS() function returns a gTTS object that does not generate speech immediately rather stores settings (customize accents, speed, preprocessing, cleaning and tokenization). 
# Actual audio is generated only when you call the save() method on this returned object, when it sends the object to Google TTS API and saves output audio
response = gTTS(
            text=text,        # Required: Text input string to convert to speech
            lang='en',        # Default = 'en') Has support of lot of languages like 'en': 'English', 'ur': 'Urdu',, 'ar': 'Arabic', 'de': 'German', 'hi': 'Hindi' and so on
            slow=False,       # Default = False means Normal speaking rate. Can be set to  True for slow speech
            tld='co.in',      # Default = 'com', that specifies US accent. Changing it gives regional accents. Top-level domain can have different values like 'com' → American,  'co.uk' → British, 'com.au' → Australian, 'co.in' → Indian
            lang_check=True,  # Default = True → Verify that 'lang' is supported
            )



# Save the audio in a file named tts.mp3
#response.save("../data/audios/gtts-generated-audio1.mp3")
with open('../data/audios/gtts-generated-audio1.mp3', 'wb') as f:
    response.write_to_fp(f)

# Play audio inside Jupyter Notebook
Audio(filename = "../data/audios/gtts-generated-audio1.mp3", autoplay=False)

In [4]:
from gtts import gTTS        
from IPython.display import Audio 

#  Input text you want to convert to speech
text = "پیارے طلباء، میں آپ سب کو جنریٹو آرٹیفیشل انٹیلی جنس کے کورس میں خوش آمدید کہتا ہوں۔ میں آپ کا انسٹرکٹر عارف بٹ ہوں۔"

# The gTTS() function returns a gTTS object that does not generate speech immediately rather stores settings (customize accents, speed, preprocessing, cleaning and tokenization). 
# Actual audio is generated only when you call the save() method on this returned object, when it sends the object to Google TTS API and saves output audio
response = gTTS(
            text=text,                 # Required: Text input string to convert to speech
            lang='ur',                 # Default = 'en') Has support of lot of languages like 'en': 'English', 'ur': 'Urdu',, 'ar': 'Arabic', 'de': 'German', 'hi': 'Hindi' and so on
            slow=False,                # Default = False means Normal speaking rate. Can be set to  True for slow speech
            tld='co.in',               # Default = 'com', that specifies US accent. Changing it gives regional accents. Top-level domain can have different values like 'com' → American,  'co.uk' → British, 'com.au' → Australian, 'co.in' → Indian
            lang_check=True,            # Default = True → Verify that 'lang' is supported
            )
# Save the generated speech to an MP3 file
response.save("../data/audios/gtts-generated-audio2.mp3")

# Play audio inside Jupyter Notebook
Audio(filename = "../data/audios/gtts-generated-audio2.mp3", autoplay=False)

### Using Open-Source Models wiht Transformers Pipeline
- You can create a TTS pipeline in Hugging Face using the `pipeline("text-to-speech", model_name)` method. Some popular TTS models are:
    - **`suno/bark-small`:** (https://huggingface.co/suno/bark-small) Lightweight TTS model from Suno.ai that generates natural-sounding English speech, optimized to run on CPU or low-VRAM machines.
    - **`facebook/mms-tts-eng`:** (https://huggingface.co/facebook/mms-tts-eng) Massively Multilingual Speech model from Facebook AI, supporting over 1100 languages, suitable for high-quality speech synthesis in many languages.
    - **`microsoft/speecht5_tts`:** (https://huggingface.co/microsoft/speecht5_tts) Transformer-based TTS model from Microsoft that produces expressive speech and can optionally use speaker embeddings for voice customization.
- The pipeline returns a waveform (numpy array) + sample rate. You can save it with soundfile and play it inline using Audio.

In [5]:
from transformers import pipeline
from IPython.display import Audio

# Load Bark TTS pipeline
tts_pipeline = pipeline("text-to-speech", model="suno/bark-small", device=-1)

# Input text (can include special commands like [clears throat], [laughs])
text = "Hello students, welcome to learning Generative AI with Arif  Butt."

# Generate speech
speech = tts_pipeline(text)

# NOTE (transformers v5): the TTS pipeline now returns a FLAT 1-D waveform.
# In v4 it was shaped (1, N) so the code indexed [0]; doing that in v5 yields a single
# float32 sample and raises "Array audio input must be a 1D or 2D array".
# Play directly without saving the audio file on disk
Audio(speech["audio"], rate=speech["sampling_rate"])

Loading weights:   0%|          | 0/542 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'min_eos_p', 'return_dict_in_generate'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:10000 for open-end generation.
[transformers] The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Both `max_new_tokens` (=768) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (http

In [7]:
from transformers import pipeline
from scipy.io.wavfile import write
from IPython.display import Audio

# Load Bark TTS pipeline
tts_pipeline = pipeline("text-to-speech", model="suno/bark-small", device=-1)

# Input text (can include special commands like [clears throat], [laughs])
text = "Let me give you a funny statement, [clears throat], I shot an elephant wearning pajamas [laughs]."

# Generate speech
speech = tts_pipeline(text)

# Save audio file on disk
# NOTE (transformers v5): waveform is a flat 1-D array now - do not index it with [0]
write('../data/audios/speech-generatedby-bark-small.wav', 
      rate=speech["sampling_rate"], 
      data=speech["audio"])

# Load autio file from disk and play in notebook
Audio('../data/audios/speech-generatedby-bark-small.wav')

Loading weights:   0%|          | 0/542 [00:00<?, ?it/s]

[transformers] The attention mask and the pad token id were not set. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.
[transformers] Setting `pad_token_id` to `eos_token_id`:10000 for open-end generation.
[transformers] Both `max_new_tokens` (=768) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=60) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://

# 3. <span style='background :lightgreen' >Sample Project (Cross-Modal Chaining)</span>

- Input: Audio from the user via Gradio microphone component.
- Processing: Send audio → Groq-hosted LLM using the ask_groq function.
- Output: Receive text from Groq → convert to speech using gTTS.
- Playback: Play generated speech in Gradio.

```
Microphone
   ↓
Gradio Audio (filepath)
   ↓
Whisper-base (ASR)
   ↓
Groq-hosted LLM
   ↓
Text response
   ↓
gTTS
   ↓
Audio output
```

In [8]:
import os
from dotenv import load_dotenv
import gradio as gr
from transformers import pipeline
from gtts import gTTS
from openai import OpenAI


# Load environment variables
load_dotenv("../keys/.env", override=True)
groq_api_key = os.getenv("GROQ_API_KEY")
# Groq client (OpenAI-compatible API)
client = OpenAI(base_url="https://api.groq.com/openai/v1", api_key=groq_api_key)

# Whisper ASR (open-source, local)
transcriber = pipeline("automatic-speech-recognition", model="openai/whisper-base", device=-1)

# Function to call Groq-hosted LLM
def ask_groq(
        user_prompt: str,
        developer_prompt: str = "You are a helpful assistant. Keep answers concise.",
        model: str = "llama-3.3-70b-versatile",
        max_output_tokens: int = 512,
        temperature: float = 0.7,
        top_p: float = 1.0,
        ):
                response = client.responses.create(
                                                    model=model,
                                                    input=[{"role": "developer", "content": developer_prompt}, {"role": "user", "content": user_prompt}],
                                                    max_output_tokens=max_output_tokens,
                                                    temperature=temperature,
                                                    top_p=top_p,
                                                    )

                return response.output_text

# Main chatbot function (audio → text → LLM → audio)
def voice_chatbot(audio_path):
    if audio_path is None:
        return "No audio received.", None
    # 1. Speech → Text (Whisper)
    transcription = transcriber(audio_path)
    user_text = transcription["text"]
    # 2. Text → Groq LLM
    bot_text = ask_groq(user_text)
    # 3. Text → Speech (gTTS)
    output_audio_path = "bot_response.mp3"
    tts = gTTS(text=bot_text, lang="en")
    tts.save(output_audio_path)
    return bot_text, output_audio_path

# Gradio UI 
with gr.Blocks() as demo:
    gr.Markdown("""
    <h1 align=center>Vice Chatbot (Whisper + Groq + gTTS)</h1>
    """)
    with gr.Row():
        mic_input = gr.Audio(
            label="Speak",
            type="filepath"   # IMPORTANT for Whisper
        )

    bot_text_output = gr.Textbox(
        label="Bot Text Response",
        interactive=False
    )

    bot_audio_output = gr.Audio(
        label="Bot Audio Response",
        type="filepath",
        interactive=False
    )

    mic_input.change(
        fn=voice_chatbot,
        inputs=mic_input,
        outputs=[bot_text_output, bot_audio_output]
    )

demo.launch(inbrowser=True)

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


# 4. <span style='background :lightgreen' >Text → Video Generation</span>
Text-to-video models transform written descriptions into dynamic moving visuals. By combining advances in diffusion and multimodal learning, these systems can generate short video clips with realistic motion, artistic styles, or cinematic effects directly from prompts.
- **Closed Source:**
    - **OpenAI Sora:** OpenAI’s text-to-video model that generates short clips (up to ~20 seconds, 1080p) from text prompts, or remixes/extends video/image inputs. (https://openai.com/sora/)
    - **RunwayML Gen-2:** A commercial video generation tool for stylized text→video outputs (known for strong visuals and creative flexibility).
    - **Pika Labs:** Proprietary system for generating animated video content via text prompts, with a focus on accessible tools for creators.
- **Open Source:**
    - **ModelScope:** An open framework / repository of video generation models and tools for academic & community use.
    - **Zeroscope:** A powerful open-source text-to-video model (v2 and XL) that generates short realistic videos from prompts, without watermarks, in aspect ratios close to 16:9. (https://zeroscopeai.com/)
    - **AnimateDiff:** Open community tools / diffusion-based pipelines to animate text-to-image outputs or generate short video motion from text/image inputs.
- **Example Prompts:**
    - “A time-lapse of a flower blooming in a garden.”
    - “A cat walking through a cyberpunk alley at night.”
- **Key Takeaway:** From text descriptions alone, modern systems can generate moving visual narratives — dynamic scenes, changing lighting & motion — bridging static imagery and video storytelling.

### Using OpenAI's `sora-2` (Text → Video)
- **Sora-2** is OpenAI's hosted text-to-video model. It is closed source and available only through the OpenAI API.
- Video generation is **asynchronous**: `client.videos.create()` returns immediately with a job whose `status` is `queued`. You then poll until the status becomes `completed`, and finally download the MP4.
- The OpenAI client provides the following methods for the Sora model:
    - **`videos.create()`:** Submits the job and returns at once. You poll `videos.retrieve(id)` yourself.
    - **`videos.create_and_poll()`:** Convenience wrapper that submits the job and blocks until it finishes. Used below.
    - **`videos.download_content()`:** Downloads the finished asset (`variant="video"` for the MP4, or `"thumbnail"` / `"spritesheet"`).
- Valid parameter values (as accepted by the installed `openai` SDK):
    - **`model`:** `sora-2` (720p) or `sora-2-pro` (higher quality, and the only one accepting the 1024x1792 / 1792x1024 sizes)
    - **`seconds`:** `"4"`, `"8"`, `"12"` &nbsp;&nbsp;**Note: this is a *string*, not an integer** - passing `seconds=4` raises an error. Default is `"4"`.
    - **`size`:** `"720x1280"` (portrait, the default), `"1280x720"` (landscape), `"1024x1792"`, `"1792x1024"`
- <span style='background:#ffe0e0'>**COST WARNING:** Unlike every other model in this notebook, Sora is billed **per second of video generated**, not per token. `sora-2` costs **\$0.10/second**, so the 4-second clip below costs **\$0.40**. `sora-2-pro` costs \$0.30-\$0.70/second. Always keep `seconds="4"` while learning, and re-run the cell only when you need to - each run is a fresh charge.</span>
- Docs: https://developers.openai.com/api/docs/guides/video-generation


In [ ]:
# Text -> Video using OpenAI's sora-2  (COST: ~$0.40 for this 4-second 720p clip)
import os
from openai import OpenAI
from IPython.display import Video

client = OpenAI(api_key=openai_api_key)

# Make sure the output folder exists (the repo ships data/audios and data/images, but not data/videos)
os.makedirs('../data/videos', exist_ok=True)

# create_and_poll() submits the job AND blocks until it is finished, so we do not have to write a polling loop.
# Expect this to take roughly 1-3 minutes - video generation is far slower than text or image generation.
video = client.videos.create_and_poll(
    model="sora-2",                 # 'sora-2' = 720p at $0.10/sec. 'sora-2-pro' is 3-7x more expensive - avoid it in class.
    prompt="A time-lapse of a red rose blooming in a sunlit garden, soft focus background, gentle camera push-in.",
    seconds="4",                    # STRING not int. Allowed: "4", "8", "12". Keep at "4" to minimise cost.
    size="1280x720",                # Landscape 720p. Other options: "720x1280" (portrait, default), and for sora-2-pro only: "1024x1792", "1792x1024"
    poll_interval_ms=5000,          # How often the SDK checks whether the job is done (5 seconds)
)

print(f"Job id     : {video.id}")
print(f"Job status : {video.status}")     # queued -> in_progress -> completed (or failed)

# The job can finish as 'failed' (e.g. the prompt was rejected by the content filter), so always check before downloading
if video.status == "completed":
    # download_content() fetches the finished asset. variant can be "video", "thumbnail", or "spritesheet"
    content = client.videos.download_content(video.id, variant="video")
    content.write_to_file('../data/videos/sora-generated-video1.mp4')
    print("Saved to ../data/videos/sora-generated-video1.mp4")
else:
    print(f"Generation failed: {getattr(video, 'error', 'no error detail returned')}")

# Play the generated video inside the notebook
Video('../data/videos/sora-generated-video1.mp4', embed=True, width=640)


### Using Open-Source `text-to-video-ms-1.7b` (Text → Video, runs locally and FREE)
- Sora is excellent but is **billed per second of video**, which adds up quickly across a whole class. The open-source alternative below runs **entirely on your own machine at zero cost** - once the weights are downloaded you can generate as many clips as you like, with no API key.
- **`ali-vilab/text-to-video-ms-1.7b`** (also known as *ModelScope Text-to-Video*) is a 1.7B-parameter latent diffusion model. It is the same family of technology as Stable Diffusion, except the UNet carries extra **temporal** layers, so it denoises a *stack of frames* together instead of a single image - which is what makes the output move coherently.
- **Why this model for a classroom?**
    - The `fp16` variant downloads only **~3.7 GB** (UNet 2.8 GB + text encoder 0.7 GB + VAE 0.2 GB), so it fits comfortably on a 16 GB laptop.
    - Newer open models are far heavier: `Wan2.1-T2V-1.3B` ships a ~29 GB repo (its T5 text encoder alone dominates) and `CogVideoX-2b` is ~14 GB - both painful to download for a class and likely to exhaust memory on 16 GB.
- **Device selection:** the code auto-detects `cuda` (NVIDIA) → `mps` (Apple Silicon) → `cpu`. On CPU a single clip can take many minutes; lower `num_inference_steps` if you are waiting too long.
- **Set expectations honestly:** the output is **256x256, soft/blurry, and about 2 seconds long** - nothing like Sora. That contrast *is* the lesson: free + local + private + non-commercial weights, versus paid + hosted + far higher fidelity.
- **Expect minutes, not seconds.** Measured end-to-end on an **Apple M1 Pro (16 GB, MPS, float32)**: about **50 seconds per denoising step**, so the 15-step clip below takes roughly **12 minutes**. Generate your demo clip *before* class rather than live. Peak memory stayed within 16 GB with no swapping.
- <span style='background:#ffe0e0'>**Apple Silicon gotcha (already handled in the code below):** running this model in `float16` on the MPS backend silently produces **all-NaN (blank) frames** - no error is raised, the generation just takes minutes and yields nothing. The notebook therefore downloads the small fp16 weight files but computes in `float32` on MPS. Do not "optimise" this back to `torch_dtype=torch.float16` on a Mac.</span>
- **Two harmless warnings you will see - tell students to ignore them:**
    1. `The TextToVideoSDPipeline has been deprecated and will not receive bug fixes ... after Diffusers version 0.33.1` - the pipeline still runs fine in 0.37.1; this is a `logger.warning`, not an error.
    2. Do **not** pass `clip_skip` to this pipeline. With `transformers` v5 the internals of `CLIPTextModel` changed, and the `clip_skip` code path in diffusers still expects the old layout, so it raises `AttributeError`. Leaving it unset (the default) takes the safe path.
- **Bandwidth warning for instructors:** at a typical connection this download takes a long time, and **30 students pulling 3.7 GB simultaneously during a lecture will not work**. Have students download it beforehand, or share a pre-populated `~/.cache/huggingface` cache.
- **License:** the weights are **CC-BY-NC 4.0** - research and teaching are fine, commercial use is not.
- HuggingFace: https://huggingface.co/ali-vilab/text-to-video-ms-1.7b


In [1]:
# Open-source Text -> Video running locally: FREE, no API key, no per-second billing.
# FIRST RUN downloads ~3.7 GB of weights and caches them under ~/.cache/huggingface (later runs are instant).
import torch
from diffusers import DiffusionPipeline

# Pick the best available accelerator: CUDA (NVIDIA) > MPS (Apple Silicon) > CPU
if torch.cuda.is_available():
    device, dtype = "cuda", torch.float16     # fp16 is fast and reliable on NVIDIA
elif torch.backends.mps.is_available():
    device, dtype = "mps", torch.float32      # see the float16 warning below
else:
    device, dtype = "cpu", torch.float32      # float16 is poorly supported on CPU

print(f"Using device: {device} (dtype={dtype})")

# TWO SEPARATE SETTINGS - this is the subtle part:
#   variant="fp16"  -> chooses WHICH FILES to DOWNLOAD (the small half-precision ones, ~3.7 GB
#                      instead of ~7.3 GB). Always keep this on: it saves bandwidth and disk.
#   torch_dtype     -> chooses the COMPUTE precision once the weights are in memory.
#
# WHY float32 ON APPLE SILICON: running this model in float16 on the MPS backend silently
# produces all-NaN frames. There is no error - generation appears to succeed, takes minutes,
# and then the saved video is blank. Verified on an M1 Pro: float16 -> NaN, float32 -> correct
# output. So we download the small fp16 files but upcast them to float32 for the actual maths.
# (Casting the VAE alone to float32 does NOT work - it raises a Half/float dtype mismatch.)
pipe = DiffusionPipeline.from_pretrained(
    "ali-vilab/text-to-video-ms-1.7b",
    variant="fp16",          # small download regardless of compute dtype
    torch_dtype=dtype,       # float32 on MPS/CPU, float16 on CUDA
)
pipe = pipe.to(device)

# Trades a little speed for a noticeably lower peak-memory footprint. Recommended on 16 GB machines.
pipe.enable_attention_slicing()

# OPTIONAL SPEED-UP: this model ships with a DDIMScheduler. Swapping in DPMSolverMultistep
# usually reaches comparable quality in fewer denoising steps, so you can drop num_inference_steps
# further. Uncomment to experiment (verify the quality yourself before relying on it in class):
# from diffusers import DPMSolverMultistepScheduler
# pipe.scheduler = DPMSolverMultistepScheduler.from_config(pipe.scheduler.config)

print(f"Pipeline loaded: {type(pipe).__name__}")


Using device: mps (dtype=torch.float16)


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/372 [00:00<?, ?it/s]

The TextToVideoSDPipeline has been deprecated and will not receive bug fixes or feature updates after Diffusers version 0.33.1. 


Pipeline loaded: TextToVideoSDPipeline


In [ ]:
# Generate the clip, then encode the frames into an MP4
import os, subprocess, tempfile, shutil
import numpy as np
from PIL import Image
from IPython.display import Video

# NOTE: diffusers ships an `export_to_video` helper, but it needs either `imageio`+`imageio-ffmpeg`
# or `opencv-python` as a PYTHON package - it never shells out to the ffmpeg binary on your PATH.
# Neither is installed here, so calling it raises an ImportError whose message advises
# "pip install opencv-python". IGNORE that advice: this project is managed by uv, so the correct
# command would be `uv add imageio imageio-ffmpeg` (a bare pip install would not update uv.lock).
# We avoid the dependency entirely: dump the frames as PNGs and let the ffmpeg BINARY - which this
# notebook already uses for Whisper - mux them into an MP4.
def to_pil(frame):
    """Convert one frame to a PIL image.

    The pipeline hands back float32 arrays in the range 0..1, NOT uint8. Passing those straight
    to Image.fromarray raises 'TypeError: Cannot handle this data type', so rescale to 0..255 first.
    """
    if isinstance(frame, Image.Image):
        return frame
    arr = np.asarray(frame)
    if arr.dtype != np.uint8:                      # float32 in 0..1 -> uint8 in 0..255
        arr = (np.clip(arr, 0, 1) * 255).round().astype(np.uint8)
    return Image.fromarray(arr)


def frames_to_video(frames, output_path, fps=8):
    """Encode a list of frames (PIL Images or numpy arrays) into an MP4 using the ffmpeg binary."""
    tmpdir = tempfile.mkdtemp()
    try:
        for i, frame in enumerate(frames):
            to_pil(frame).convert("RGB").save(os.path.join(tmpdir, f"f_{i:04d}.png"))
        os.makedirs(os.path.dirname(output_path) or ".", exist_ok=True)
        subprocess.run(
            ["ffmpeg", "-loglevel", "error", "-y", "-framerate", str(fps),
             "-i", os.path.join(tmpdir, "f_%04d.png"),
             "-c:v", "libx264", "-pix_fmt", "yuv420p",
             # libx264 requires even width/height; this rounds an odd dimension down by one pixel
             "-vf", "scale=trunc(iw/2)*2:trunc(ih/2)*2",
             output_path],
            check=True,
        )
    finally:
        shutil.rmtree(tmpdir)       # always clean up the temp PNGs, even if ffmpeg fails
    return output_path


prompt = "A time-lapse of a red rose blooming in a sunlit garden"

# Keep these SMALL - here the cost is minutes of your own compute rather than dollars.
# MEASURED on an Apple M1 Pro (16 GB, MPS, float32, attention slicing): ~50 seconds PER STEP, so:
#      10 steps ~ 8 min   |   15 steps ~ 12 min   |   25 steps ~ 21 min   |   50 steps ~ 42 min
# ALWAYS pass num_inference_steps explicitly: this pipeline DEFAULTS TO 50, which would take
# about 24 minutes and stall a live lecture.
#   num_frames          -> clip length (16 frames at 8 fps = 2 seconds)
#   num_inference_steps -> denoising steps; more = better quality but linearly slower
#   height / width      -> must be multiples of 8; 256x256 keeps both memory and time down
result = pipe(
    prompt,
    num_frames=16,
    num_inference_steps=15,
    height=256,
    width=256,
    generator=torch.Generator(device="cpu").manual_seed(42),   # fixed seed => reproducible clip
)

frames = result.frames[0]      # .frames is a BATCH of videos; [0] is our single video (a list of frames)
print(f"Generated {len(frames)} frames")

out_path = frames_to_video(frames, '../data/videos/opensource-generated-video1.mp4', fps=8)
print(f"Saved to {out_path}")

# Also save an animated GIF. diffusers' `export_to_gif` is pure PIL (no ffmpeg, no cv2, no imageio),
# so it always works - a handy fallback if the ffmpeg call above fails on a student's machine.
from diffusers.utils import export_to_gif
gif_path = export_to_gif([to_pil(f) for f in frames], '../data/videos/opensource-generated-video1.gif', fps=8)
print(f"Saved to {gif_path}")

Video(out_path, embed=True, width=384)


  0%|          | 0/15 [00:00<?, ?it/s]

# 5. <span style='background :lightgreen' >Video → Text (Video Understanding)</span>

Video-to-text models extend visual understanding to dynamic sequences by analyzing both spatial and temporal information. They can interpret actions, events, and context across frames, enabling applications such as video summarization, content moderation, and accessibility.
- **Closed Source:**
    - **GPT-4o:** Multimodal model by OpenAI with strong capabilities to interpret video + audio + vision + text for temporal reasoning.
    - **Gemini Pro Vision:** Google DeepMind’s high-capacity vision-enabled version of Gemini for interpreting video content.
    - **Claude 3.5 Sonnet:** Anthropic’s multimodal model with enhanced video understanding and reasoning.
- **Open Source:**
    - **Video-LLaVA:** Unified visual-language model that aligns image & video inputs into the same representation space, allowing mixed image+video question answering.
    - **Chat-UniVi:** Model that uses unified visual tokens for both images and videos, enabling efficient temporal + spatial reasoning.
    - **Video-XL:** Extra-Long Vision-Language model for hour-scale video understanding, compressing visual input while preserving fine detail. 
- **Example Prompts:**
    - Upload cooking video → “Summarize the recipe and cooking steps.”
    - Upload sports clip → “Describe the key moments and player actions.”
- **Key Takeaway:** Models can analyze temporal visual sequences (i.e. video), track changes over time, understand context, and extract meaning from moving images in addition to static frames.


### Using OpenAI's `gpt-4o` (Video → Text)
- Today's chat models do **not** accept an MP4 file directly. The standard technique is to treat a video as **a sequence of still images**:
    1. Decode the video and grab one frame every *N* frames (**frame sampling**).
    2. Base64-encode each sampled frame as a JPEG.
    3. Send them all as `input_image` blocks inside a **single** request - the model then reasons over the whole sequence and can describe motion and ordering, not just one still.
- **Why sample instead of sending every frame?** A 4-second clip at 30fps is 120 frames. Each image costs tokens, so sending all of them is both slow and expensive. Sampling every 10th-25th frame keeps cost low while preserving the storyline.
- <span style='background:#ffe0e0'>**COST NOTE:** Cost scales with the *number of frames*, not the length of the video. The cell below sends only 6 frames, which is a few cents. Raise `MAX_FRAMES` carefully.</span>
- The frames below are extracted with **ffmpeg**, which you already installed for Whisper earlier in this notebook - so there is no extra Python dependency to install. (The OpenAI cookbook uses OpenCV / `cv2` for this; ffmpeg avoids the extra package.)
- **Audio:** this technique is vision-only. To also understand what is *said* in a video, extract the audio track and run it through Whisper (Section 1), then pass that transcript alongside the frames.


In [ ]:
# Step 1: Extract sampled frames from a video file using ffmpeg, and base64-encode them
import base64, glob, os, subprocess

VIDEO_PATH  = '../data/videos/sora-generated-video1.mp4'   # Reusing the clip generated by Sora above
FRAMES_DIR  = '../data/videos/frames'
MAX_FRAMES  = 6          # Keep this small: every frame is billed as an image. 6 frames over a 4-second clip = ~1.5 fps

os.makedirs(FRAMES_DIR, exist_ok=True)
for old in glob.glob(f'{FRAMES_DIR}/*.jpg'):    # clear any frames left over from a previous run
    os.remove(old)

# fps=1.5 tells ffmpeg to emit 1.5 frames per second of video; -vframes caps the total number written.
# scale=512:-1 shrinks each frame to 512px wide (height auto) - smaller images cost fewer tokens.
subprocess.run(
    ['ffmpeg', '-loglevel', 'error', '-i', VIDEO_PATH,
     '-vf', 'fps=1.5,scale=512:-1', '-vframes', str(MAX_FRAMES),
     f'{FRAMES_DIR}/frame_%03d.jpg'],
    check=True
)

# Read each JPEG off disk and base64-encode it, exactly as we did for images in Part-I of this lecture
frame_paths = sorted(glob.glob(f'{FRAMES_DIR}/frame_*.jpg'))
base64_frames = []
for path in frame_paths:
    with open(path, 'rb') as f:
        base64_frames.append(base64.b64encode(f.read()).decode('utf-8'))

print(f"Extracted and encoded {len(base64_frames)} frames from {VIDEO_PATH}")

# Preview the sampled frames so students can see what the model is actually being shown
from IPython.display import Image, display
for path in frame_paths[:3]:
    display(Image(filename=path, width=220))


In [ ]:
# Step 2: Send all sampled frames in ONE request and ask the model to describe the video
from openai import OpenAI

client = OpenAI(api_key=openai_api_key)

response = client.responses.create(
    model="gpt-4o",                  # Any vision-capable model works here (gpt-4o, gpt-4o-mini, gpt-4.1, ...)
    input=[{
        "role": "user",
        "content": [
            {"type": "input_text",
             "text": ("These are sequential frames sampled from a single short video, in chronological order. "
                      "Describe what happens across the clip, including any motion or camera movement. "
                      "Answer in 3-4 sentences.")},
            # Unpack every frame into its own input_image block inside the SAME message,
            # so the model sees the whole sequence at once and can reason about the ORDER of events.
            *[{"type": "input_image", "image_url": f"data:image/jpeg;base64,{frame}"}
              for frame in base64_frames],
        ],
    }],
)

print(response.output_text)

# Token usage - note how the frames dominate the input token count
print(f"\nInput tokens: {response.usage.input_tokens}  |  Output tokens: {response.usage.output_tokens}")
